# <center>Laboratorio 9: Benchmark de Carga y Modelos con Spotify 🎵</center>

<center><strong>MDS7202: Laboratorio de Programación Científica para Ciencia de Datos</strong></center>

---

### Cuerpo Docente

- Profesores: Pablo Badilla y Diego Cortez
- Auxiliares: Valentina Rojas y Melanie Peña
- Ayudantes: Javiera Arévalo, Tamara Carrasco e Ignacio Reyes

### Equipo: SUPER IMPORTANTE - notebooks sin nombre no serán revisados

- Nombre de alumno 1: Jiale Chen
- Nombre de alumno 2:

---

### Reglas

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente serán respondidos por este medio.
- Prohibido copiar.
- Uso de LLM (Copilot, Claude, Cursor, etc.) restringido a consultas, documentación y corrección de errores.

# Temas a tratar

- Lectura eficiente de datos en formato Parquet.
- Optimización del uso de memoria mediante conversión de tipos de datos.
- Paralelización de operaciones I/O con `ThreadPoolExecutor`.
- Comparación de implementaciones de predicción: Python, NumPy, Numba, pandas y Polars.
- Entrenamiento de modelos con RandomForestRegressor y efecto de `n_jobs`.
- Orquestación de pipelines de datos con Apache Airflow.

# Objetivos principales del laboratorio

- Cargar datos de canciones de Spotify desde archivos Parquet y optimizar su representación en memoria.
- Comparar el tiempo de lectura de archivos en serie vs. en paralelo.
- Analizar el impacto de distintas implementaciones (Python puro, NumPy, Numba, pandas, Polars) en el tiempo de predicción de un modelo lineal.
- Entrenar un RandomForestRegressor que prediga la valencia de canciones, comparando el efecto de la paralelización del entrenamiento.
- Orquestar el pipeline completo (carga + entrenamiento) usando Apache Airflow.

> Instalamos e importamos las librerías necesarias 🎸

In [2]:
!uv pip install pandas pyarrow lightgbm scikit-learn plotly apache-airflow polars numba

/bin/bash: line 1: uv: command not found


In [3]:
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from pathlib import Path

import numba
import numpy as np
import pandas as pd
import plotly.express as px
import polars as pl
from sklearn.metrics import root_mean_squared_error
from sklearn.model_selection import train_test_split

DATA_DIR = Path("data")

# 1. Carga y Optimización de Datos con Parquet

Los datos que usaremos en este laboratorio corresponden a un dataset de canciones de Spotify almacenado en **20 archivos Parquet** (`batch_01.parquet` … `batch_20.parquet`), con un total de 200 000 canciones y 24 columnas que incluyen características de audio, metadatos y la letra completa de cada canción.

A continuación trabajaremos en dos aspectos fundamentales de la carga de datos en la práctica:
1. **Optimizar el uso de memoria** ajustando los tipos de datos de las columnas.
2. **Reducir el tiempo de carga** paralelizando la lectura de archivos.

## 1.1 Exploración y Optimización de Tipos de Datos [1 Punto]

Cuando cargamos datos con pandas, los tipos inferidos por defecto no siempre son los más eficientes. Por ejemplo, un entero que siempre cabe en 16 bits se almacena por defecto como `int64` (64 bits), usando 4 veces más memoria de la necesaria. Lo mismo ocurre con flotantes y con columnas categóricas almacenadas como strings.

**Código dado** — funciones de carga:

In [4]:
def load_batch(path: str) -> pd.DataFrame:
    """Lee un único archivo Parquet y retorna un DataFrame."""
    return pd.read_parquet(path)


def load_all_serial(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    """Lee todos los archivos Parquet de data_dir en serie y los concatena."""
    paths = sorted(data_dir.glob("*.parquet"))
    if n_batches is not None:
        paths = paths[:n_batches]
    return pd.concat([load_batch(str(p)) for p in paths], ignore_index=True)

**TO-DO [0.3 Puntos]:**
- [ ] Ejecutar `load_all_serial` sobre todos los batches y explorar el DataFrame resultante (`.dtypes`, `.memory_usage(deep=True)`).
- [ ] Aplicar las siguientes conversiones a un nuevo DataFrame (copia del originalmente cargado `df_opt`):
  - `float64` → `float32`: columnas de audio features (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, `tempo`, `avg_artist_popularity`).
  - `int64` → `int16`: columnas `key`, `mode`.
  - `int64` → `int32`: columnas `year`, `popularity`, `duration_ms`, `total_artist_followers`.
- [ ] Comparar el uso de memoria antes y después con un gráfico de barras usando Plotly (código dado).

In [5]:
# Escribe aquí tu código
df = load_all_serial(DATA_DIR)

# Explorar tipos de datos y uso de memoria
display(df.dtypes)
display(df.memory_usage(deep=True))

df_opt = df.copy()  # .astype(...)

df_opt = df_opt.astype(
    {
        # float64 a float32
        "danceability": "float32",
        "energy": "float32",
        "loudness": "float32",
        "speechiness": "float32",
        "acousticness": "float32",
        "instrumentalness": "float32",
        "liveness": "float32",
        "valence": "float32",
        "tempo": "float32",
        "avg_artist_popularity": "float32",
        # int64 a int16
        "key": "int16",
        "mode": "int16",
        # int64 a int32
        "year": "int32",
        "popularity": "int32",
        "duration_ms": "int32",
        "total_artist_followers": "int32",
    }
)

# Verificar los tipos después de la conversión
display(df_opt.dtypes)

# Compara el uso de memoria antes y después con un gráfico de barras
mem_before = df.memory_usage(deep=True).sum() / 1024**2
mem_after = df_opt.memory_usage(deep=True).sum() / 1024**2

px.bar(
    x=["Antes", "Después"],
    y=[mem_before, mem_after],
    labels={"x": "Estado", "y": "Uso de Memoria (MiB)"},
    title=f"Uso de memoria: {mem_before:.1f} MiB → {mem_after:.1f} MiB ({(1 - mem_after / mem_before) * 100:.1f}% reducción)",
).show()

id                         object
name                       object
album_name                 object
artists                    object
danceability              float64
energy                    float64
key                         int64
loudness                  float64
mode                        int64
speechiness               float64
acousticness              float64
instrumentalness          float64
liveness                  float64
valence                   float64
tempo                     float64
duration_ms                 int64
lyrics                     object
year                        int64
genre                      object
popularity                  int64
total_artist_followers      int64
avg_artist_popularity     float64
artist_ids                 object
niche_genres               object
dtype: object

Index                           132
id                         14200000
name                       13638614
album_name                 13876175
artists                    24000000
danceability                1600000
energy                      1600000
key                         1600000
loudness                    1600000
mode                        1600000
speechiness                 1600000
acousticness                1600000
instrumentalness            1600000
liveness                    1600000
valence                     1600000
tempo                       1600000
duration_ms                 1600000
lyrics                    284216252
year                        1600000
genre                      10839938
popularity                  1600000
total_artist_followers      1600000
avg_artist_popularity       1600000
artist_ids                 24000000
niche_genres               24000000
dtype: int64

id                         object
name                       object
album_name                 object
artists                    object
danceability              float32
energy                    float32
key                         int16
loudness                  float32
mode                        int16
speechiness               float32
acousticness              float32
instrumentalness          float32
liveness                  float32
valence                   float32
tempo                     float32
duration_ms                 int32
lyrics                     object
year                        int32
genre                      object
popularity                  int32
total_artist_followers      int32
avg_artist_popularity     float32
artist_ids                 object
niche_genres               object
dtype: object

### Preguntas [0.7 Puntos]

1. ¿Qué es el formato **Parquet**? ¿Qué ventajas tiene sobre CSV para datos analíticos? ¿Qué es *columnar storage* y por qué acelera las consultas que solo leen algunas columnas?
2. ¿Qué es **Apache Arrow**? ¿Cómo se relaciona con Parquet y con pandas internamente? ¿Qué ganas al usar `pd.read_parquet` en vez de `pd.read_csv`?
3. ¿Por qué existe `float32` si `float64` es más preciso? ¿En qué contextos esa pérdida de precisión es irrelevante?
4. ¿Cuándo **no** conviene reducir la precisión de un tipo numérico? ¿Qué riesgos concretos existen?
5. ¿Existe alguna alternativa a pandas para trabajar con estos datos de forma más eficiente en memoria? (menciona al menos dos)
6. ¿Cuánto se redujo el uso de memoria en total (en MiB y en %)? ¿Era esperable ese resultado? ¿Por qué no se redujo tanto como podría esperarse?
7. ¿Qué pasaría si intentaras reducir `valence` a `float16`? ¿Qué riesgo existiría para el modelo entrenado en la sección 2?

**Escribe tus respuestas aquí...**

1- Parquet es un formato binario para almacenar datos tabulares de forma eficiente. A diferencia de CSV, conserva los tipos de datos, utiliza compresión y ocupa menos espacio, por lo que suele ofrecer lecturas más rápidas en análisis de grandes datasets.

El *columnar storage* significa que los datos se guardan por columnas en lugar de por filas. Esto permite leer únicamente las columnas necesarias para una consulta, reduciendo la cantidad de datos leídos, el uso de memoria y el tiempo de procesamiento.

2- Apache Arrow es un estándar para representar datos tabulares en memoria de forma columnar, permitiendo compartirlos entre librerías y lenguajes con pocas conversiones y copias.

Parquet almacena datos comprimidos en disco, mientras que Arrow los representa eficientemente en memoria. Pandas puede usar `pyarrow` para leer archivos Parquet y convertirlos en DataFrames.

Usar `pd.read_parquet` suele ser más rápido y consumir menos espacio que `pd.read_csv`, porque conserva los tipos de datos, usa compresión y permite leer solo las columnas necesarias. CSV guarda todo como texto y requiere analizar y convertir los valores durante la carga.

3- `float32` existe porque usa la mitad de memoria que `float64` y puede acelerar cálculos, transferencias de datos y procesamiento en grandes volúmenes. Aunque ofrece menos precisión, sigue siendo suficiente para muchos datos reales.

La pérdida de precisión suele ser irrelevante en variables como medidas de sensores, características de audio, imágenes, porcentajes o datos usados en modelos de machine learning, donde el ruido de los datos es mayor que el error introducido por `float32`. `float64` es más adecuado cuando se requieren cálculos científicos muy precisos, acumulaciones numéricas largas o resultados financieros sensibles.

4- No conviene reducir la precisión cuando los cálculos requieren alta exactitud, como en simulaciones científicas, cálculos financieros, modelos numéricos sensibles, acumulaciones largas o valores muy grandes o muy pequeños.

Los principales riesgos son errores de redondeo, pérdida de decimales significativos, desbordamiento, resultados inestables y acumulación de pequeñas diferencias a lo largo de muchas operaciones. También puede ocurrir que dos valores distintos terminen representándose como iguales, afectando comparaciones, métricas o decisiones del modelo.

5- Sí. Dos alternativas son **Polars** y **PyArrow**. Polars utiliza procesamiento columnar, ejecución paralela y evaluación diferida, por lo que suele consumir menos memoria y ser más rápido que pandas. PyArrow permite trabajar directamente con datos en formato Arrow y Parquet, reduciendo conversiones y copias innecesarias.

También existen opciones como **Dask**, que permiten procesar datos más grandes que la memoria disponible mediante particiones o consultas directas sobre archivos Parquet.

6- El uso de memoria se redujo de **414.2 MiB** a **401.3 MiB**, es decir, una disminución de **12.9 MiB**, equivalente a aproximadamente **3.1 %**.

El resultado era esperable porque solo se optimizaron algunas columnas numéricas. Aunque convertir `float64` a `float32`, `int64` a `int32` o `int16` reduce el tamaño de esas columnas, gran parte de la memoria total probablemente corresponde a columnas de texto, como nombres, artistas, metadatos y letras de canciones.

Por eso la reducción total no fue tan grande: las columnas de tipo `object` o string suelen ocupar mucha más memoria que las columnas numéricas y no fueron optimizadas.

7- Reducir `valence` de `float32` a `float16` disminuiría todavía más el uso de memoria, pero también reduciría significativamente la precisión de sus valores.

El principal riesgo es que `valence` es la variable objetivo del modelo. Al usar `float16`, valores cercanos podrían redondearse al mismo número, introduciendo error en las etiquetas de entrenamiento. Esto podría disminuir la precisión del modelo, alterar el RMSE y limitar su capacidad para aprender diferencias pequeñas entre canciones. En este caso, el ahorro de memoria probablemente no compensaría la pérdida de información.

In [6]:
# **IMPORTANTE**: Una vez contestada la pregunta, ejecutar esta celda para liberar memoria.
df_opt = None

## 1.2 Lectura en Serie vs. Paralelo [1 Punto]

Cuando se trabaja con múltiples archivos, la lectura **en paralelo** puede reducir el tiempo total al aprovechar que la espera de I/O (disco/red) no bloquea al procesador. En Python, la clase `ThreadPoolExecutor` del módulo `concurrent.futures` permite lanzar múltiples hilos para ejecutar operaciones de forma concurrente.

**TO-DO: [0.3 Puntos]**
- [ ] Implementar `load_all_parallel` usando `ThreadPoolExecutor`.
- [ ] Medir con `%timeit` ambas versiones sobre todos los batches.
- [ ] Generar un gráfico de línea (Plotly) con los tiempos para 2, 4, 6, …, 20 archivos, con series `Serial` y `Paralelo`.

In [7]:
# Escribe aquí tu código
def load_all_parallel(data_dir: Path, n_batches: int | None = None) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))

    if n_batches is not None:
        paths = paths[:n_batches]

    with ThreadPoolExecutor() as executor:
        dataframes = list(executor.map(load_batch, map(str, paths)))

    return pd.concat(dataframes, ignore_index=True)

In [8]:
%timeit load_all_serial(DATA_DIR)
%timeit load_all_parallel(DATA_DIR)

702 ms ± 97.1 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
426 ms ± 24.2 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


**Benchmark:** mide tiempos para 2, 4, 6, ..., 20 archivos y grafica


In [9]:
@dataclass
class ReadMeasurement:
    n_files: int
    time_sec: float
    version: str


measurements: list[ReadMeasurement] = []

for n in range(2, 21):
    t0 = time.perf_counter()
    load_all_serial(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Serial"))

    t0 = time.perf_counter()
    load_all_parallel(DATA_DIR, n_batches=n)
    measurements.append(ReadMeasurement(n, time.perf_counter() - t0, "Paralelo"))

df_times = pd.DataFrame(measurements)
px.line(
    df_times,
    x="n_files",
    y="time_sec",
    color="version",
    markers=True,
    title="Tiempo de lectura: Serial vs Paralelo",
    labels={"n_files": "Número de archivos", "time_sec": "Tiempo (s)"},
).show()

### Preguntas  [0.7 Puntos]

1. ¿Qué significa que una operación sea **I/O-bound** vs **CPU-bound**? ¿A cuál categoría pertenece la lectura de archivos desde disco?
2. ¿Qué es el **GIL** (*Global Interpreter Lock*) de CPython? ¿Por qué existe? ¿Qué problema resuelve y qué limitación introduce?
3. ¿Por qué usamos Python si tiene el GIL? ¿Qué ganamos al usarlo como lenguaje de *pegamento* entre librerías de alto rendimiento (NumPy, Arrow, PyTorch…)?
4. ¿Cuándo conviene usar `ThreadPoolExecutor` vs `ProcessPoolExecutor`? ¿Cuál usarías si la operación fuera puramente CPU-bound?
5. ¿Qué overhead introduce crear un pool de threads? ¿Qué pasaría si los archivos fueran muy pequeños (p.ej. 1 KB cada uno)?
6. ¿Se observó mejora con la lectura paralela? ¿A partir de cuántos archivos empieza a ser notable?
7. ¿Por qué el speedup obtenido **no es igual** al número de threads disponibles? ¿Qué factores lo limitan?

**Escribe tus respuestas aquí...**

1- Una operación **I/O-bound** pasa la mayor parte del tiempo esperando operaciones de entrada/salida, como leer archivos, acceder a una base de datos o recibir datos por red. Una operación **CPU-bound** dedica la mayor parte del tiempo a realizar cálculos intensivos en el procesador.

La lectura de archivos desde disco es principalmente **I/O-bound**, porque el programa suele esperar a que el sistema de almacenamiento entregue los datos. Por eso puede beneficiarse del uso de varios hilos, ya que mientras uno espera una lectura, otro puede avanzar con otro archivo.

2- El **GIL** (*Global Interpreter Lock*) es un mecanismo de CPython que permite que solo un hilo ejecute código Python a la vez dentro de un mismo proceso.

Existe para simplificar la gestión de memoria y proteger estructuras internas del intérprete frente a accesos simultáneos, evitando errores y condiciones de carrera.

La limitación es que los hilos no aceleran tareas **CPU-bound** escritas en Python puro, porque no pueden ejecutar bytecode en paralelo. Sin embargo, sí pueden ser útiles en tareas **I/O-bound**, ya que el GIL suele liberarse mientras un hilo espera operaciones de disco, red o entrada/salida.

3- Python se sigue utilizando porque es fácil de leer, desarrollar y mantener, además de contar con un ecosistema muy amplio para análisis de datos, machine learning y automatización.

Aunque el GIL limita el paralelismo del código Python puro, muchas librerías como NumPy, Arrow o PyTorch ejecutan las operaciones intensivas en código compilado en C, C++ o CUDA, donde el GIL puede liberarse. Python actúa como lenguaje de *pegamento*: permite coordinar estas librerías de alto rendimiento con poco código, manteniendo productividad sin renunciar a velocidad.

4- `ThreadPoolExecutor` conviene para tareas **I/O-bound**, como leer archivos, hacer solicitudes de red o consultar bases de datos, porque los hilos pueden avanzar mientras otros esperan.

`ProcessPoolExecutor` conviene para tareas **CPU-bound**, ya que usa procesos independientes y evita la limitación del GIL, permitiendo ejecutar cálculos realmente en paralelo.

Si la operación fuera puramente CPU-bound, usaría `ProcessPoolExecutor`, de lo contrario `ThreadPoolExecutor`.

5- Crear un pool de hilos introduce un pequeño *overhead*: hay que crear y administrar los threads, repartir las tareas, coordinar sus resultados y realizar cambios de contexto entre ellos.

Si los archivos fueran muy pequeños, por ejemplo de 1 KB, ese costo podría ser mayor que el tiempo real de lectura. En ese caso, la versión paralela podría rendir igual o incluso peor que la serial. La paralelización suele ser más útil cuando cada tarea de I/O tiene un costo suficiente para compensar ese overhead. 

6- Sí, la lectura paralela mostró una mejora clara frente a la lectura serial. En todo el rango medido, la versión paralela tarda menos y la diferencia aumenta a medida que se leen más archivos.

La mejora empieza a ser notable aproximadamente desde 4 o 5 archivos. Con pocos archivos la diferencia es pequeña, pero a partir de ese punto la lectura paralela se separa de forma más visible. Con 20 archivos, el tiempo baja de aproximadamente **0.76 s** en serial a **0.47 s** en paralelo, lo que representa una reducción cercana al **38 %**.

7- El *speedup* no es igual al número de threads porque no todo el proceso puede ejecutarse en paralelo. Parte del tiempo se dedica a crear y coordinar los hilos, concatenar los DataFrames y gestionar resultados, tareas que siguen teniendo un costo secuencial.

También influyen la velocidad máxima del disco, el ancho de banda de memoria, la descompresión de Parquet, la competencia entre hilos por los mismos recursos y el overhead de los cambios de contexto. Además, si todos los hilos intentan leer al mismo tiempo, el dispositivo de almacenamiento puede saturarse.

Por estas razones, agregar más threads produce mejoras cada vez menores y, a partir de cierto punto, incluso puede empeorar el rendimiento.

# 2. Predicción de Valencia

La columna `valence` de Spotify mide el **positivismo musical** de una canción: valores cercanos a 1 indican canciones alegres y eufóricas, mientras que valores cercanos a 0 corresponden a canciones tristes o melancólicas. En esta sección analizaremos distintas formas de realizar predicciones con un modelo de regresión lineal ya entrenado, y luego entrenaremos un modelo más complejo.

## 2.1 Regresión Lineal a Mano [1.5 Puntos]

Antes de entrenar un modelo completo, veremos cómo **la elección de implementación** afecta drásticamente el rendimiento de predicción. Usaremos un modelo de regresión lineal pre-entrenado cuyos coeficientes ya están dados, e implementaremos la predicción usando cinco enfoques distintos: Python puro, NumPy, Numba (JIT), pandas y Polars.

**Código dado — carga de datos y parámetros del modelo:**

In [10]:
# Carga de datos y preparación del split
df_train = load_all_serial(DATA_DIR, n_batches=20)

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]

X = df_train[PARAM_COLS + ["key", "mode", "genre"]]
y = df_train["valence"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
# Parámetros del modelo lineal pre-entrenado (dados)
params = {
    "danceability": 0.7718203106411208,
    "energy": 0.4252134896942928,
    "loudness": -0.008319917439312445,
    "speechiness": -0.24543273088867107,
    "acousticness": 0.10440236785191129,
    "instrumentalness": -0.11203723673701874,
    "liveness": 0.023790522969698424,
    "tempo": 0.0007885690378158087,
    "duration_ms": -4.31739613602265e-07,
    "year": -0.0036043842721972985,
}
intercept = 6.948154825159983

params_vals = list(params.values())
params_arr = np.array(params_vals, dtype=np.float32)

**Código dado — las 5 implementaciones de predicción:**

Analiza cómo cada implementación aborda el mismo problema y presta atención a las diferencias en legibilidad, concisión y (como verás en el benchmark) rendimiento.

In [12]:
def linear_regression_predict(X: np.ndarray, params: list[float]) -> list[float]:
    """Predicción con loop Python puro."""
    preds = []
    for row in X:
        val = intercept
        for j, w in enumerate(params):
            val += row[j] * w
        preds.append(val)
    return preds


def linear_regression_predict_numpy(X: np.ndarray, params: list[float]) -> np.ndarray:
    """Predicción vectorizada con NumPy."""
    return (X * np.array(params)).sum(axis=1) + intercept


@numba.njit
def linear_regression_predict_numba(X: np.ndarray, params: np.ndarray) -> np.ndarray:
    """Predicción con Numba JIT (loop compilado a código máquina)."""
    n = X.shape[0]
    preds = np.empty(n)
    for i in range(n):
        val = intercept
        for j in range(len(params)):
            val += X[i, j] * params[j]
        preds[i] = val
    return preds


def linear_regression_predict_pandas(X: pd.DataFrame, params: list[float]) -> pd.Series:
    """Predicción vectorizada con pandas (dot product)."""
    return X.dot(pd.Series(params, index=X.columns)) + intercept


def linear_regression_predict_polars(X: pl.DataFrame, params: list[float]) -> pl.Series:
    """Predicción vectorizada con Polars (expresiones lazy)."""
    weights = dict(zip(X.columns, params, strict=False))
    expr = pl.lit(intercept)
    for col, w in weights.items():
        expr = expr + pl.col(col) * w
    return X.select(expr.alias("pred"))["pred"]

**Código dado — Benchmark de las 5 implementaciones:**

In [13]:
@dataclass
class TimeMeasurement:
    time_took: float
    iteration: int
    version: str


time_measurements: list[TimeMeasurement] = []

ranges = [10, 50, 100, 250, 500, 750, 1000, *range(1001, len(X_test) + 1, 1000)]

for it in ranges:
    X_np = X_test[PARAM_COLS].iloc[:it].to_numpy(dtype=np.float32)
    X_pd = X_test[PARAM_COLS].iloc[:it]
    X_pl = pl.from_pandas(X_pd)

    for name, fn, args in [
        ("Python", linear_regression_predict, (X_np, params_vals)),
        ("NumPy", linear_regression_predict_numpy, (X_np, params_vals)),
        ("Numba-JIT", linear_regression_predict_numba, (X_np, params_arr)),
        ("Pandas", linear_regression_predict_pandas, (X_pd, params_vals)),
        ("Polars", linear_regression_predict_polars, (X_pl, params_vals)),
    ]:
        t0 = time.perf_counter()
        fn(*args)
        time_measurements.append(TimeMeasurement(time.perf_counter() - t0, it, name))

df_bench = pd.DataFrame(time_measurements)

# Gráfico 1: tiempos absolutos
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
).show()

# Gráfico 2: tiempos absolutos (en log)
px.line(
    df_bench,
    x="iteration",
    y="time_took",
    color="version",
    markers=True,
    title="Tiempos de predicción según implementación (en escala logarítmica)",
    labels={"iteration": "Número de filas", "time_took": "Tiempo (s)"},
    log_y=True,
).show()

# Gráfico 3: speedup relativo respecto a Python puro
pivot = df_bench.pivot(index="iteration", columns="version", values="time_took")
for col in ["NumPy", "Numba-JIT", "Pandas", "Polars"]:
    pivot[col] = pivot["Python"] / pivot[col]
pivot["Python"] = 1.0

melted = pivot.reset_index().melt(
    id_vars=["iteration"],
    value_vars=["Python", "NumPy", "Numba-JIT", "Pandas", "Polars"],
    value_name="speedup",
)
px.line(
    melted,
    x="iteration",
    y="speedup",
    color="version",
    markers=True,
    title="Speedup relativo respecto a Python puro",
    labels={"iteration": "Número de filas", "speedup": "Speedup (×)"},
).show()

### Preguntas [1.5 Puntos]

  1. ¿Qué es la vectorización en NumPy? ¿Cómo puede ejecutar operaciones sobre arrays sin loops de Python explícitos?
  2. ¿Qué es JIT (Just-In-Time compilation)? ¿Qué hace el decorador @numba.njit? ¿Qué significa el modo nopython?
  3. ¿Por qué Numba es más lento en la primera ejecución? ¿Qué es el warm-up de JIT y cómo lo manejamos en el benchmark?
  4. ¿Qué es Polars y cuáles son sus principales características como librería de datos? ¿Para qué escenarios fue diseñada y por qué ha ganado popularidad como alternativa a pandas?
  5. ¿En qué se diferencia Polars de pandas a nivel de implementación (lenguaje, modelo de ejecución, manejo de memoria)?
  6. ¿Por qué pandas puede ser más lento que NumPy aun usando operaciones vectorizadas internamente?
  7. ¿Qué son las instrucciones SIMD (Single Instruction Multiple Data)? ¿Cómo contribuyen a la aceleración de NumPy y Polars?
  8. ¿Cuándo conviene usar Numba sobre NumPy? ¿Y Polars sobre pandas para operaciones numéricas?
  9. ¿Cuál implementación fue la más rápida en tu medición? ¿Era esperable ese resultado?
  10. ¿Se observa diferencia notable entre pandas y NumPy? ¿Por qué pandas puede ser más lento o más rápido?
  11. ¿A partir de cuántas filas empieza a ser evidente la ventaja de NumPy/Numba sobre Python puro?
  12. ¿Polars fue más eficiente que pandas en tu medición? Verifica la versión de pandas instalada (pd.__version__) y comenta si crees que la versión influye en el resultado.
  13. ¿Por qué Numba puede igualar o superar a NumPy para loops numéricos simples?
  14. El benchmark excluye el costo de convertir datos a NumPy/Polars (la conversión ocurre fuera del timing). ¿Cómo cambiaría el resultado si incluyeras ese costo? ¿En qué escenarios de
  producción ese costo no existiría?
  15.  Si tuvieras que realizar esta predicción sobre 100 millones de filas en un servidor de producción, ¿qué implementación elegirías y por qué? ¿Cambiaría tu respuesta si dispusieras de
  una GPU?

**Escribe tus respuestas aquí...**

Observaciones: 

Los resultados muestran que **Python puro es la implementación más lenta**, porque realiza las operaciones elemento por elemento mediante bucles interpretados. Su tiempo crece de forma clara al aumentar el número de filas.

**NumPy, pandas y Polars** son mucho más rápidos porque ejecutan operaciones vectorizadas en código compilado. Polars presenta, en general, mejores tiempos que NumPy y pandas, mientras que pandas tiene un pequeño overhead asociado a sus índices y estructuras internas.

**Numba-JIT es la implementación más rápida** después de la compilación inicial. Sin embargo, la primera ejecución es muy lenta porque incluye el tiempo necesario para compilar la función a código máquina.

1- La vectorización en NumPy consiste en aplicar una operación a un array completo sin escribir loops explícitos en Python. Por ejemplo, una multiplicación y suma sobre miles de filas puede expresarse con operaciones como `X * params` o `X @ params`.

NumPy puede hacerlo porque ejecuta esas operaciones internamente en código compilado, principalmente C y rutinas optimizadas de álgebra lineal. Así evita el costo de interpretar cada iteración en Python y trabaja directamente sobre bloques contiguos de memoria, lo que mejora considerablemente el rendimiento.

2- La compilación **JIT** (*Just-In-Time*) consiste en traducir una función a código máquina durante la ejecución del programa, normalmente la primera vez que se llama. Esto añade un costo inicial de compilación, pero las llamadas posteriores suelen ser mucho más rápidas.

El decorador `@numba.njit` compila la función con Numba y evita ejecutar sus operaciones mediante el intérprete de Python. Los loops y cálculos numéricos pasan a ejecutarse como código compilado.

El modo **nopython** significa que Numba debe compilar toda la función sin depender de objetos ni del intérprete de Python. Esto ofrece el mejor rendimiento, pero exige usar tipos y operaciones compatibles con Numba. `@numba.njit` equivale a usar `@numba.jit(nopython=True)`.

3- Numba es más lento en la primera ejecución porque debe analizar los tipos de datos y compilar la función a código máquina; este costo inicial se conoce como *warm-up* del JIT. Después, reutiliza la versión compilada y las ejecuciones siguientes son mucho más rápidas. Para que el benchmark sea justo, se debe ejecutar la función una vez antes de medir, evitando que el tiempo de compilación quede incluido en los resultados (no se hizo en este caso).

4- Polars es una librería para manipular datos tabulares basada en Apache Arrow y escrita en Rust. Se caracteriza por su ejecución rápida, uso eficiente de memoria, procesamiento multihilo y soporte para evaluación *lazy*, que optimiza las consultas antes de ejecutarlas. Fue diseñada para análisis de datos medianos y grandes, especialmente en tareas con muchas transformaciones, agregaciones o archivos Parquet. Ha ganado popularidad como alternativa a pandas porque suele ofrecer mejor rendimiento, menor consumo de memoria y una API expresiva para operaciones columnares.

5- Polars está implementado principalmente en **Rust**, mientras que pandas está construido sobre **Python, C y NumPy**. Polars utiliza un modelo de ejecución columnar, paralelo y con soporte para evaluación *lazy*, lo que le permite optimizar una consulta completa antes de ejecutarla. Pandas suele ejecutar las operaciones de forma inmediata y muchas transformaciones se procesan paso a paso. En memoria, Polars usa estructuras basadas en Apache Arrow, que son compactas y favorecen operaciones sin copias innecesarias, mientras que pandas puede tener mayor sobrecarga por sus índices, objetos de Python y tipos `object`.

6- Pandas puede ser más lento que NumPy porque, además de ejecutar operaciones numéricas, debe gestionar índices, nombres de columnas, alineación automática, tipos de datos y valores nulos. NumPy trabaja directamente con arrays homogéneos y con menos sobrecarga, mientras que pandas añade una capa de abstracción y verificaciones que facilitan el análisis de datos, pero aumentan el tiempo de ejecución.

7- Las instrucciones **SIMD** (*Single Instruction Multiple Data*) permiten que el procesador aplique una misma operación a varios valores al mismo tiempo, en lugar de procesarlos uno por uno. NumPy y Polars aprovechan estas instrucciones mediante código compilado y operaciones vectorizadas, lo que acelera cálculos como sumas, multiplicaciones, filtros y agregaciones sobre columnas completas. Esto reduce el número de instrucciones necesarias y mejora el uso del procesador, especialmente en grandes volúmenes de datos.

8- Conviene usar **Numba sobre NumPy** cuando el cálculo incluye loops, condiciones o lógica personalizada que no se puede expresar fácilmente con operaciones vectorizadas. Numba compila ese código y puede acercarlo al rendimiento de C. **Polars sobre pandas** conviene cuando se trabaja con datasets grandes, muchas transformaciones, agregaciones o archivos Parquet, ya que aprovecha ejecución paralela, procesamiento columnar y menor uso de memoria. Para operaciones numéricas simples sobre arrays homogéneos, NumPy suele seguir siendo la opción más directa y eficiente.

9- La implementación más rápida fue **Numba-JIT**, una vez completado el *warm-up* de compilación. Era un resultado esperable porque Numba convierte los loops numéricos en código máquina y evita el overhead del intérprete de Python. La primera ejecución fue más lenta por el costo de compilación, pero en las siguientes obtuvo el mayor *speedup* frente a Python puro.

10- Sí, se observa una diferencia, aunque no tan grande como frente a Python puro. NumPy suele ser más rápido porque trabaja directamente con arrays homogéneos y tiene menos sobrecarga. Pandas puede ser más lento debido a la gestión de índices, nombres de columnas, alineación y tipos de datos. Sin embargo, en algunos casos puede acercarse o incluso superar a NumPy si la operación está muy optimizada internamente o si el costo de convertir previamente los datos a un array de NumPy es significativo.

11- La ventaja de NumPy y Numba sobre Python puro empieza a ser evidente desde aproximadamente **500 a 1.000 filas**. A partir de ese tamaño, el tiempo de Python crece mucho más rápido, mientras que NumPy y Numba mantienen tiempos muy bajos gracias a la vectorización y al código compilado.

12- Sí, en la medición **Polars fue más eficiente que pandas**. La versión instalada es **pandas 2.3.3**, una versión reciente con mejoras de rendimiento y soporte más sólido para Apache Arrow. Aun así, la versión influye solo parcialmente: Polars suele mantener ventaja por su implementación en Rust, ejecución multihilo, modelo columnar y menor sobrecarga en operaciones numéricas.

13- Numba puede igualar o superar a NumPy en loops numéricos simples porque compila el bucle completo a código máquina, elimina el costo del intérprete de Python y puede optimizar accesos a memoria y operaciones repetitivas. NumPy también usa código compilado, pero algunas expresiones crean arrays temporales o ejecutan varias operaciones separadas; Numba puede fusionar todo el cálculo en un solo loop, reduciendo copias y overhead.

14- Si se incluyera el costo de convertir los datos a NumPy o Polars, sus tiempos aumentarían y la ventaja frente a pandas podría reducirse, especialmente para datasets pequeños o para una sola predicción. Ese costo no existiría, o sería mínimo, en producción cuando los datos ya llegan en el formato requerido, se mantienen en memoria entre varias operaciones, se leen directamente como arrays o DataFrames de Polars, o la conversión se realiza una sola vez y luego se reutiliza en muchas predicciones.

15- Para realizar la predicción sobre **100 millones de filas** en un servidor de producción elegiría **Numba** o una implementación directa con **NumPy**, procesando los datos por bloques para no saturar la memoria. Numba sería una buena opción si la lógica incluye loops o cálculos personalizados, porque compila la función a código máquina y evita arrays temporales. Si la predicción puede expresarse como una multiplicación matricial simple, NumPy también sería muy eficiente y más sencillo de mantener.

Con una **GPU**, usaría una librería como **CuPy o PyTorch**, siempre que los datos puedan mantenerse en la memoria de la GPU y procesarse en lotes grandes. La GPU puede acelerar mucho este tipo de operaciones masivas, pero el beneficio disminuye si el tiempo de transferir los datos entre la RAM y la GPU es mayor que el tiempo de cálculo.

In [14]:
pd.__version__

'2.3.3'

### 2.2 Entrenamiento y Comparación de `n_jobs` [0.5 Puntos]

Ahora entrenaremos un modelo más complejo: un **RandomForestRegressor** que usa las características de audio más una codificación del género musical para predecir `valence`. Compararemos el efecto de paralelizar el entrenamiento con el parámetro `n_jobs`.

**Código dado — pipeline encapsulado** (no modificar):

In [15]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def build_pipeline(n_jobs: int = 1) -> Pipeline:
    # En producción este pipeline usaría LGBMRegressor; aquí usamos RandomForest
    # para ilustrar el efecto de n_jobs de forma más pronunciada.
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        (
                            "numerical",
                            "passthrough",
                            PARAM_COLS,
                        ),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )

In [ ]:
# Entrena con n_jobs=1 y mide el tiempo
pipeline_1 = build_pipeline(n_jobs=1)
t0 = time.perf_counter()
pipeline_1.fit(X_train, y_train)
time_1job = time.perf_counter() - t0

# Entrena con n_jobs=-1 y mide el tiempo
pipeline_all = build_pipeline(n_jobs=-1)
t0 = time.perf_counter()
pipeline_all.fit(X_train, y_train)
time_all_jobs = time.perf_counter() - t0

# Calcula RMSE de ambos modelos
rmse_1 = root_mean_squared_error(y_test, pipeline_1.predict(X_test))
rmse_all = root_mean_squared_error(y_test, pipeline_all.predict(X_test))

print(f"n_jobs=1  → tiempo: {time_1job:.1f}s | RMSE: {rmse_1:.4f}")
print(f"n_jobs=-1 → tiempo: {time_all_jobs:.1f}s | RMSE: {rmse_all:.4f}")

In [ ]:
# Gráficos de tiempos y RMSE
df_perf = pd.DataFrame(
    {
        "configuracion": ["n_jobs=1", "n_jobs=-1"],
        "tiempo_s": [time_1job, time_all_jobs],
        "rmse": [rmse_1, rmse_all],
    }
)

px.bar(
    df_perf,
    x="configuracion",
    y="tiempo_s",
    title="Tiempo de entrenamiento según n_jobs",
    labels={"tiempo_s": "Tiempo (s)", "configuracion": "Configuración"},
    text_auto=".1f",
).show()

px.bar(
    df_perf,
    x="configuracion",
    y="rmse",
    title="RMSE según n_jobs",
    labels={"rmse": "RMSE", "configuracion": "Configuración"},
    text_auto=".4f",
).show()

### Preguntas [0.5 Puntos]

1. ¿Qué hace el parámetro `n_jobs` en RandomForest (y en general en scikit-learn)?
2. **¿Por qué aquí sí funciona el paralelismo real sin el problema del GIL?** (Pista: RandomForest en scikit-learn usa joblib con backend de procesos o threads nativos.)
3. ¿Cuánto mejoró el tiempo con `n_jobs=-1`? 
4. ¿Fue proporcional al número de CPUs disponibles en tu máquina? ¿Por qué no?
5. ¿Hubo diferencia en RMSE entre ambas versiones? ¿Era esperable? ¿Por qué?

**Escribe tus respuestas aquí...**

1- El parámetro `n_jobs` controla cuántos núcleos o procesos de CPU usa scikit-learn para ejecutar las partes paralelizables de un algoritmo. En `RandomForestRegressor`, permite entrenar y evaluar varios árboles al mismo tiempo, ya que cada árbol puede construirse de manera independiente. Con `n_jobs=1` se usa un solo núcleo, mientras que `n_jobs=-1` utiliza todos los núcleos disponibles. Esto puede reducir mucho el tiempo de entrenamiento, aunque no cambia la calidad del modelo, como se observa en el mismo RMSE para ambas configuraciones.

2- Aquí sí existe paralelismo real porque `RandomForestRegressor` delega el trabajo a código compilado y a `joblib`, que puede distribuir el entrenamiento de los árboles entre varios procesos o hilos nativos. Cada árbol se entrena de forma casi independiente, por lo que distintos núcleos pueden trabajar al mismo tiempo. Además, gran parte del cálculo ocurre fuera del intérprete de Python, donde el GIL no bloquea la ejecución, por eso `n_jobs=-1` puede aprovechar varios núcleos y reducir considerablemente el tiempo de entrenamiento.

3- El tiempo de entrenamiento bajó de **159 s** con `n_jobs=1` a **15.1 s** con `n_jobs=-1`. Esto representa una reducción de aproximadamente **143.9 s**, equivalente a cerca de **90.5 %**, y un *speedup* de alrededor de **10.5 veces**.

4- No fue proporcional: con **20 procesos efectivos**, el *speedup* observado fue de aproximadamente **10.5×**, no de 20×. Esto ocurre porque parte del entrenamiento sigue siendo secuencial y porque existen costos de coordinación, reparto de trabajo, acceso compartido a memoria y combinación de resultados. Además, los procesos compiten por recursos como el ancho de banda de memoria y la caché, por lo que el rendimiento no escala de forma lineal.

5- No hubo diferencia en el RMSE: ambos modelos obtuvieron aproximadamente **0.1651**. Era esperable porque `n_jobs` solo cambia la cantidad de recursos utilizados durante el entrenamiento, no el algoritmo ni sus parámetros. Además, al usar `random_state=42`, ambos modelos generan el mismo bosque y producen las mismas predicciones.

In [ ]:
from joblib import effective_n_jobs

n_jobs = -1
print("Procesos efectivos:", effective_n_jobs(n_jobs))

Procesos efectivos: 20


# 3. Orquestación del Pipeline con Apache Airflow

En producción, los pipelines de datos y ML rara vez se ejecutan a mano desde un notebook. Se necesita:
- **Automatización**: que el pipeline corra periódicamente (diariamente, por hora…).
- **Dependencias**: que el entrenamiento solo comience si la carga de datos terminó exitosamente.
- **Monitoreo y reintentos**: que si una tarea falla, el sistema lo registre y reintente.

**Apache Airflow** resuelve exactamente esto. Define pipelines como **DAGs** (*Directed Acyclic Graphs*), donde cada nodo es una **tarea** y las aristas definen dependencias.

| Concepto | Descripción |
|----------|-------------|
| **DAG** | Grafo Dirigido Acíclico que representa el pipeline completo |
| **Operator** | Unidad de trabajo (`PythonOperator`, `BashOperator`, …) |
| **Task** | Instancia de un Operator dentro de un DAG |
| **XCom** | Mecanismo para pasar datos pequeños entre tareas |
| **schedule** | Expresión cron que indica cuándo ejecutar el DAG |

### Setup local


En la carpeta del Lab:

```bash
export AIRFLOW_HOME=$(pwd)
airflow db migrate          # inicializa la base de datos de metadata
# Ver la contraseña. Si no se en un comienzo, ejecutar airflow standalone, parar el proceso y luego ejecutar nuevamente este comando. 
cat $AIRFLOW_HOME/simple_auth_manager_passwords.json.generated 
airflow standalone       # levanta scheduler + webserver en http://localhost:8080
```

Los DAGs deben guardarse en `./dags`.

## 3.1 Implementación del DAG

**TO-DO [0.8 Puntos]:**
- [ ] Implementar `task_load_data_fn`: cargar 5 batches en paralelo, guardar en disco como Parquet y pasar la ruta a la siguiente tarea usando XCom.
- [ ] Implementar `task_train_model_fn`: recuperar la ruta de XCom, cargar el DataFrame, preparar X e y, entrenar `build_pipeline(n_jobs=-1)` e imprimir el tiempo.
- [ ] Definir la dependencia entre tareas (`load_data >> train_model`).

El siguiente bloque es el template que debes completar en tu celda de respuesta.

In [ ]:
# %%writefile ~/airflow/dags/spotify_pipeline_dag.py

# from pathlib import Path
# import time
# import pandas as pd
# from concurrent.futures import ThreadPoolExecutor

# from airflow import DAG
# from airflow.operators.python import PythonOperator
# from datetime import datetime

# from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
# from sklearn.preprocessing import OneHotEncoder
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import train_test_split


# DATA_DIR = Path("/RUTA/ABSOLUTA/A/Labs/Lab9_v2/data")  # AJUSTA esta ruta
# OUTPUT_PATH = Path("/tmp/spotify_data.parquet")

# PARAM_COLS = [
#     "danceability",
#     "energy",
#     "loudness",
#     "speechiness",
#     "acousticness",
#     "instrumentalness",
#     "liveness",
#     "tempo",
#     "duration_ms",
#     "year",
# ]


# # ── Funciones auxiliares (dadas) ─────────────────────────────────────────────


# def load_batch(path: str) -> pd.DataFrame:
#     return pd.read_parquet(path)


# def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
#     paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
#     with ThreadPoolExecutor(max_workers=None) as executor:
#         dfs = list(executor.map(load_batch, [str(p) for p in paths]))
#     return pd.concat(dfs, ignore_index=True)


# def build_pipeline(n_jobs: int = -1) -> Pipeline:
#     return Pipeline(
#         [
#             (
#                 "column_transformer",
#                 ColumnTransformer(
#                     [
#                         ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
#                         ("numerical", "passthrough", PARAM_COLS),
#                     ]
#                 ),
#             ),
#             ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
#         ]
#     )


# # ── Funciones de las tareas de Airflow ───────────────────────────────────────


# def task_load_data_fn(**context):
#     """
#     Carga 5 batches de datos en paralelo y guarda el resultado en disco.
#     TODO: implementa esta función.
#     - Usa load_all_parallel para cargar los datos.
#     - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
#     - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
#     """
#     ...


# def task_train_model_fn(**context):
#     """
#     Carga los datos desde disco y entrena el pipeline.
#     TODO: implementa esta función.
#     - Recupera la ruta del archivo desde XCom.
#     - Lee el DataFrame desde esa ruta.
#     - Prepara X e y, realiza el split 80/20.
#     - Entrena build_pipeline(n_jobs=-1).
#     - Imprime el tiempo de entrenamiento.
#     """
#     ...


# # ── Definición del DAG ────────────────────────────────────────────────────────

# with DAG(
#     dag_id="spotify_pipeline",
#     start_date=datetime(2026, 1, 1),
#     schedule=None,
#     catchup=False,
#     tags=["mds7202", "spotify"],
# ) as dag:
#     load_data = PythonOperator(
#         task_id="load_data",
#         python_callable=task_load_data_fn,
#     )

#     train_model = PythonOperator(
#         task_id="train_model",
#         python_callable=task_train_model_fn,
#     )

#     # TODO: define la dependencia entre tareas (load_data debe ejecutarse antes que train_model)
#     ...


In [ ]:
from pathlib import Path

dag_dir = Path.home() / "airflow" / "dags"
dag_dir.mkdir(parents=True, exist_ok=True)

print(dag_dir)

/home/bot/airflow/dags


In [ ]:
%%writefile ~/airflow/dags/spotify_pipeline_dag.py
# Escribe aquí tu código (copia el template y completa los TODOs)
from pathlib import Path
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from airflow import DAG
from airflow.operators.python import PythonOperator
from datetime import datetime

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split


DATA_DIR = Path("/home/bot/MDS7202/labs/lab_9/data")
OUTPUT_PATH = Path("/home/bot/MDS7202/labs/lab_9/spotify_data.parquet")

PARAM_COLS = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "tempo",
    "duration_ms",
    "year",
]


# ── Funciones auxiliares (dadas) ──────────────────────────────────────────────────────


def load_batch(path: str) -> pd.DataFrame:
    return pd.read_parquet(path)


def load_all_parallel(data_dir: Path, n_batches: int = 5) -> pd.DataFrame:
    paths = sorted(data_dir.glob("*.parquet"))[:n_batches]
    with ThreadPoolExecutor(max_workers=None) as executor:
        dfs = list(executor.map(load_batch, [str(p) for p in paths]))
    return pd.concat(dfs, ignore_index=True)


def build_pipeline(n_jobs: int = -1) -> Pipeline:
    return Pipeline(
        [
            (
                "column_transformer",
                ColumnTransformer(
                    [
                        ("ohe", OneHotEncoder(handle_unknown="ignore"), ["key", "mode", "genre"]),
                        ("numerical", "passthrough", PARAM_COLS),
                    ]
                ),
            ),
            ("random_forest", RandomForestRegressor(n_jobs=n_jobs, random_state=42)),
        ]
    )


# ── Funciones de las tareas de Airflow ───────────────────────────────────────


def task_load_data_fn(**context):
    """
    Carga 5 batches de datos en paralelo y guarda el resultado en disco.
    TODO: implementa esta función.
    - Usa load_all_parallel para cargar los datos.
    - Guarda el DataFrame resultante en OUTPUT_PATH (formato parquet).
    - Usa XCom para pasar la ruta del archivo a la siguiente tarea.
    """
    df = load_all_parallel(DATA_DIR, n_batches=5)

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(OUTPUT_PATH, index=False)

    context["ti"].xcom_push(
        key="data_path",
        value=str(OUTPUT_PATH),
    )

    print(f"Datos cargados: {df.shape}")
    print(f"Archivo guardado en: {OUTPUT_PATH}")


def task_train_model_fn(**context):
    """
    Carga los datos desde disco y entrena el pipeline.
    TODO: implementa esta función.
    - Recupera la ruta del archivo desde XCom.
    - Lee el DataFrame desde esa ruta.
    - Prepara X e y, realiza el split 80/20.
    - Entrena build_pipeline(n_jobs=-1).
    - Imprime el tiempo de entrenamiento.
    """
    data_path = context["ti"].xcom_pull(
        task_ids="load_data",
        key="data_path",
    )

    df = pd.read_parquet(data_path)

    X = df[PARAM_COLS + ["key", "mode", "genre"]]
    y = df["valence"]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
    )

    pipeline = build_pipeline(n_jobs=-1)

    t0 = time.perf_counter()
    pipeline.fit(X_train, y_train)
    elapsed = time.perf_counter() - t0

    print(f"Tiempo de entrenamiento: {elapsed:.2f} segundos")


# ── Definición del DAG ────────────────────────────────────────────────────────


with DAG(
    dag_id="spotify_pipeline",
    start_date=datetime(2026, 1, 1),
    schedule=None,
    catchup=False,
    tags=["mds7202", "spotify"],
) as dag:
    load_data = PythonOperator(
        task_id="load_data",
        python_callable=task_load_data_fn,
    )

    train_model = PythonOperator(
        task_id="train_model",
        python_callable=task_train_model_fn,
    )

    load_data >> train_model

Overwriting /home/bot/airflow/dags/spotify_pipeline_dag.py


Una vez guardado el archivo, ejecuta el DAG con:

```bash
airflow standalone
```


### Pega aquí el output de las steps del DAG

- Step 1: 
...


- Step 2:
...

### Preguntas [1.2 Puntos]

1. ¿Qué es un **DAG**? ¿Qué significa que sea *Directed* (dirigido) y *Acyclic* (acíclico)? ¿Por qué importa la propiedad acíclica en un pipeline de datos?
2. ¿Qué es **Apache Airflow**? ¿Para qué tipo de problemas está diseñado y cuál es su unidad mínima de trabajo?
3. ¿Qué son los **Operators**? ¿Qué diferencia hay entre `PythonOperator` y `BashOperator`? ¿Cuándo usarías cada uno?
4. ¿Qué es **XCom** en Airflow? ¿Cómo funciona internamente (¿dónde se almacena?)? ¿Por qué **no** es adecuado para pasar DataFrames grandes entre tareas?
5. ¿Qué alternativa concreta usaste para pasar el DataFrame entre `load_data` y `train_model`? ¿Cuál sería la alternativa recomendada en producción (S3, GCS, DVC…)?
6. ¿Qué es el parámetro `schedule` de un DAG? ¿Cómo lo configurarías para que corra todos los días a las 3 AM?
7. ¿Qué diferencia hay entre Airflow y otras herramientas como **Prefect**, **Dagster**, **Luigi**, **Kubeflow**? ¿Cuál es la principal crítica que se le hace a Airflow?
8. ¿Por qué conviene orquestar el pipeline en Airflow en vez de simplemente ejecutar un script Python end-to-end?
9. ¿Qué pasa si `load_data` falla a mitad de camino? ¿Airflow reintenta automáticamente? ¿Cómo controlarías el número máximo de reintentos?
10. ¿Qué ventaja tiene que las tareas estén separadas (carga y entrenamiento) vs. una sola tarea monolítica, desde el punto de vista de debugging y eficiencia?
11. ¿Cómo podemos alertar si es que algún paso falla? ¿O si la pipeline se ejecuta correctamente?
12. En un pipeline de producción real, ¿qué otras tareas añadirías al DAG?

**Escribe tus respuestas aquí...**

# Conclusión

Eso ha sido todo para el lab de hoy. Recuerda que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda, no dudes en contactarnos por el foro de U-Cursos.

<p align="center">
  <img src="https://media.giphy.com/media/l0HlBO7eyXzSZkJri/giphy.gif" width="300">
</p>